In [1]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_groq import ChatGroq

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

# Using Llama3.1-8b
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)

C:\Users\Meghana Veeramallu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

problem = "How can I get my 5-year-old to eat vegetables?"

# Step 1: The Branch Generator
prompt_branch = ChatPromptTemplate.from_template(
    "Problem: {problem}. Give me one unique, creative solution. Solution {id}:"
)

branches = RunnableParallel(
    sol1=prompt_branch.partial(id="1") | llm | StrOutputParser(),
    sol2=prompt_branch.partial(id="2") | llm | StrOutputParser(),
    sol3=prompt_branch.partial(id="3") | llm | StrOutputParser(),
)

# Step 2: The Judge
prompt_judge = ChatPromptTemplate.from_template(
    """
    I have three proposed solutions for: '{problem}'
    
    1: {sol1}
    2: {sol2}
    3: {sol3}
    
    Act as a Child Psychologist. Pick the most sustainable one (not bribery) and explain why.
    """
)
# Chain: Input -> Branches -> Judge -> Output
tot_chain = (
    RunnableParallel(problem=RunnableLambda(lambda x: x), branches=branches)
    | (lambda x: {**x["branches"], "problem": x["problem"]}) 
    | prompt_judge
    | llm
    | StrOutputParser()
)

print("--- Tree of Thoughts (ToT) Result ---")
print(tot_chain.invoke(problem))

--- Tree of Thoughts (ToT) Result ---
As a child psychologist, I would recommend Solution 1: Create a "Veggie Face" on their plate as the most sustainable approach to encourage a 5-year-old to eat vegetables. Here's why:

1. **Non-bribery method**: Unlike Solution 2, which involves a reward chart with stickers and potential bribing, Solution 1 doesn't rely on extrinsic motivators like rewards or privileges. This approach focuses on making mealtime more engaging and fun, which can lead to a more sustainable and intrinsic motivation for eating vegetables.

2. **Development of imagination and creativity**: By encouraging children to use their imagination and create a "Veggie Face," we foster their creativity and problem-solving skills. This can lead to a more positive association with mealtime and vegetables.

3. **Reducing anxiety and stress**: When children are involved in creating a fun and engaging mealtime experience, they are less likely to feel anxious or stressed about eating vege

In [3]:
# 1. The Generator (Divergence)
prompt_draft = ChatPromptTemplate.from_template(
    "Write a 1-sentence movie plot about: {topic}. Genre: {genre}."
)

drafts = RunnableParallel(
    draft_scifi=prompt_draft.partial(genre="Sci-Fi") | llm | StrOutputParser(),
    draft_romance=prompt_draft.partial(genre="Romance") | llm | StrOutputParser(),
    draft_horror=prompt_draft.partial(genre="Horror") | llm | StrOutputParser(),
)

# 2. The Aggregator (Convergence)
prompt_combine = ChatPromptTemplate.from_template(
    """
    I have three movie ideas for the topic '{topic}':
    1. Sci-Fi: {draft_scifi}
    2. Romance: {draft_romance}
    3. Horror: {draft_horror}
    
    Your task: Create a new Mega-Movie that combines the TECHNOLOGY of Sci-Fi, the PASSION of Romance, and the FEAR of Horror.
    Write one paragraph.
    """
)

# 3. The Chain
got_chain = (
    RunnableParallel(topic=RunnableLambda(lambda x: x), drafts=drafts)
    | (lambda x: {**x["drafts"], "topic": x["topic"]}) 
    | prompt_combine
    | llm
    | StrOutputParser()
)

print("--- Graph of Thoughts (GoT) Result ---")
print(got_chain.invoke("Time Travel"))

--- Graph of Thoughts (GoT) Result ---
"Echoes of Eternity" is a heart-pounding, mind-bending time-travel thriller that combines the cutting-edge technology of a sci-fi epic with the all-consuming passion of a romance and the creeping dread of a horror classic. Dr. Emma Taylor, a brilliant physicist, has invented a device that allows her to communicate with her past self, but with each attempt, she awakens a dark presence that begins to manipulate the timeline, erasing her loved ones and forcing her to relive the same pivotal moments in her life. As Emma navigates the treacherous landscape of time, she must confront her deepest fears and desires, all while racing against the clock to prevent a catastrophic future that threatens to destroy everything she holds dear. But with each iteration, the lines between reality and nightmare blur, and Emma realizes that the only way to save her loved ones and her own sanity is to confront the dark force head-on, even if it means sacrificing her own